In [ ]:
!pip install toml
import warnings
warnings.filterwarnings("ignore")
import toml
import pandas as pd
import numpy as np
import joblib
import pickle
import psycopg2

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from xgboost import XGBRegressor
Leer secretos
secrets = toml.load("secrets.toml")["postgres"]

USER = secrets["USER"]
PASSWORD = secrets["PASSWORD"]
HOST = secrets["HOST"]
PORT = secrets["PORT"]
DBNAME = secrets["DBNAME"]
Cargar dataset
df = pd.read_csv(
    "dataset/residuos_dataset.csv"
)
Limpieza
df = df.dropna()

df = df[
    df['QRESIDUOS_MUN'] > 0
]
Variables
X = df[[
    'POB_TOTAL',
    'POB_URBANA',
    'POB_RURAL',
    'PERIODO'
]]

y = df['QRESIDUOS_MUN']
Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
Modelo
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42
)

model.fit(X_train, y_train)
Evaluación
pred = model.predict(X_test)

r2 = r2_score(y_test, pred)

print("R2:", r2)
Guardar modelo
joblib.dump(
    model,
    "components/residuos_model.pkl"
)
Guardar info modelo
model_info = {
    "feature_names": list(X.columns)
}

with open(
    "components/residuos_model_info.pkl",
    "wb"
) as f:

    pickle.dump(model_info, f)
Generar registros y guardar en Supabase
model = joblib.load(
    "components/residuos_model.pkl"
)

with open(
    "components/residuos_model_info.pkl",
    "rb"
) as f:

    model_info = pickle.load(f)

feature_names = model_info["feature_names"]
Generar registros
np.random.seed(42)

data = []

for _ in range(10):

    record = [
        np.random.randint(1000, 100000),
        np.random.randint(500, 90000),
        np.random.randint(100, 20000),
        np.random.randint(2018, 2025)
    ]

    data.append(record)
Guardar en Supabase
try:

    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )

    cursor = connection.cursor()

    for record in data:

        prediction = float(
            model.predict([record])[0]
        )

        columns = ", ".join(feature_names)

        placeholders = ", ".join(
            ["%s"] * (len(record) + 1)
        )

        sql = f'''
        INSERT INTO pc_ml_residuos
        ({columns}, prediction)
        VALUES ({placeholders})
        '''

        cursor.execute(
            sql,
            record + [prediction]
        )

    connection.commit()

    cursor.close()
    connection.close()

    print(
        "✅ Registros guardados en Supabase"
    )

except Exception as e:

    print(
        f"❌ Error: {e}"
    )
